In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import datetime as dt
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/e-commerce-customer-for-behavior-analysis/ecommerce_customer_data_large.csv
/kaggle/input/e-commerce-customer-for-behavior-analysis/ecommerce_customer_data_custom_ratios.csv


In [2]:
df=pd.read_csv("/kaggle/input/e-commerce-customer-for-behavior-analysis/ecommerce_customer_data_large.csv")

In [3]:
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.000,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.000,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.000,John Rivera,31,Female,0
3,44605,2023-01-17 13:14:36,Electronics,396,3,937,Cash,31,0.000,John Rivera,31,Female,0
4,44605,2021-05-01 11:29:27,Books,259,4,2598,PayPal,31,1.000,John Rivera,31,Female,0


In [4]:
df.isnull().sum()

Customer ID                  0
Purchase Date                0
Product Category             0
Product Price                0
Quantity                     0
Total Purchase Amount        0
Payment Method               0
Customer Age                 0
Returns                  47382
Customer Name                0
Age                          0
Gender                       0
Churn                        0
dtype: int64

In [5]:
df.shape

(250000, 13)

#### We don't need the returned products, we can remove them from our dataset.

In [6]:
df = df[df["Returns"] != 1.000]

In [7]:
df["Purchase Date"].max()

'2023-09-13 18:42:49'

In [8]:
df = df.rename(columns={"Purchase Date": "Purchase_Date"})
df = df.rename(columns={"Customer ID": "Customer_ID"})
df = df.rename(columns={"Total Purchase Amount": "Total_Purchase_Amount"})
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
today_date = dt.datetime(2023, 9, 15)


### The monetary value that a customer will bring to a company throughout their relationship and communication with the company is known as Customer Lifetime Value (CLV). 

### **Why is this important?**

### If we can measure the future value our customer will provide, we can both regulate our relationship with them and adopt a more value-added approach for the medium to long term within our company. This will also play a significant role in determining the budget allocated for marketing activities.

In [9]:
cltv_c = df.groupby('Customer_ID').agg({'Customer_ID': lambda Customer_ID: Customer_ID.value_counts(),
                                        'Quantity': lambda x: x.sum(),
                                        'Total_Purchase_Amount': lambda x: x.sum()})

cltv_c.columns = ['Total Transaction', 'Total Unit', 'Total Price']


### • **CLTV** = (Customer Value / Churn Rate) × Profit Margin

### • **Customer Value** = Average Order Value * Purchase Frequency

### • **Average Order Value** = Total Price / Total Transaction

### • **Purchase Frequency** = Total Transaction / Total Number of Customers

### • **Churn Rate** = 1 - Repeat Rate

### • **Repeat Rate** = The rate of customers who have made multiple purchases divided by the total number of customers.

### • **Profit Margin** = Total Price * 0.10

In [10]:
cltv_c["Average Order Value"] = cltv_c["Total Price"] / cltv_c["Total Transaction"]

In [11]:
cltv_c["Purchase Frequency"] = cltv_c["Total Transaction"] / cltv_c.shape[0]

In [12]:
cltv_c['Profit Margin'] = cltv_c['Total Price'] * 0.10

In [13]:
repeat_rate = cltv_c[cltv_c["Total Transaction"] > 1].shape[0] / cltv_c.shape[0]

churn_rate = 1 - repeat_rate

### Let's calculate the CLTV scores and segment these scores.

In [14]:
cltv_c['Customer Value'] = cltv_c['Average Order Value'] * cltv_c["Purchase Frequency"]

In [15]:
cltv_c["CLTV"] = (cltv_c["Customer Value"] / churn_rate) * cltv_c["Profit Margin"]

cltv_c.sort_values(by="CLTV", ascending=False).head()

,Total Transaction,Total Unit,Total Price,Average Order Value,Purchase Frequency,Profit Margin,Customer Value,CLTV
Customer_ID,,,,,,,,
47165,13,41,38527,2963.615,0.000,3852.700,0.811,19456.413
28656,11,41,38428,3493.455,0.000,3842.800,0.809,19356.550
44664,10,31,37410,3741.000,0.000,3741.000,0.788,18344.581
48382,13,36,37208,2862.154,0.000,3720.800,0.783,18147.008
9831,10,29,37137,3713.700,0.000,3713.700,0.782,18077.818


In [16]:
cltv_c["Segment"] = pd.qcut(cltv_c["CLTV"], 4, labels=["D", "C", "B", "A"])

In [17]:
cltv_c.sort_values(by="CLTV",ascending=False)

,Total Transaction,Total Unit,Total Price,Average Order Value,Purchase Frequency,Profit Margin,Customer Value,CLTV,Segment
Customer_ID,,,,,,,,,
47165,13,41,38527,2963.615,0.000,3852.700,0.811,19456.413,A
28656,11,41,38428,3493.455,0.000,3842.800,0.809,19356.550,A
44664,10,31,37410,3741.000,0.000,3741.000,0.788,18344.581,A
48382,13,36,37208,2862.154,0.000,3720.800,0.783,18147.008,A
9831,10,29,37137,3713.700,0.000,3713.700,0.782,18077.818,A
...,...,...,...,...,...,...,...,...,...
27968,1,4,137,137.000,0.000,13.700,0.003,0.246,D
2709,1,1,136,136.000,0.000,13.600,0.003,0.242,D
39631,1,4,125,125.000,0.000,12.500,0.003,0.205,D
